<a href="https://colab.research.google.com/github/Amrutha-jit-acc/iiit-h-mata/blob/main/kannadatocsv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import csv
import os

def get_grade_from_filename(filename):
    # Looks for numbers 1-10 in the filename
    match = re.search(r'(?:Class|Grade|Std|Standard|Part)?[\s_-]*(\d{1,2})', filename, re.IGNORECASE)
    if match:
        return match.group(1)
    return "Unknown"

def clean_text(text):
    # Removes common header/footer noise found in KTBS textbooks
    noise = [
        "PUBLISHED", "NOT TO BE", "KTBS", "PDF", "START OF FILE",
        "Government of Karnataka", "Page", "_______", "======="
    ]
    for n in noise:
        text = text.replace(n, "")
    return text.strip()

def process_kannada_file(input_file, output_file):
    rows = []
    grade = get_grade_from_filename(input_file)
    print(f"Processing {input_file} for Grade: {grade}...")

    current_section = "General/Textbook Content"

    # regex for Kannada/English numbers (1., 1), ೧., ೧))
    # Also handles Roman numerals often used for sections (I., II., III.)
    number_pattern = r'^(\d+|[\u0C66-\u0C6F]+|[IVX]+)[\.\)]'

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Broad keywords to categorize questions from Class 1 to 10
    header_patterns = {
        "Match the Following": [r"ಹೊಂದಿಸಿ", r"ಹೊಂದಿಸಿ ಬರೆಯಿರಿ"],
        "Fill in the Blanks": [r"ಬಿಟ್ಟ ಸ್ಥಳ", r"ಖಾಲಿಬಿಟ್ಟ", r"ಸೂಕ್ತ ಪದ", r"ತುಂಬಿರಿ"],
        "True or False": [r"ಸರಿ", r"ತಪ್ಪು", r"ಹೌದು", r"ಅಲ್ಲ"],
        "MCQ": [r"ಆರಿಸಿ ಬರೆಯಿರಿ", r"ಸರಿಯಾದ ಉತ್ತರ", r"ಆಯ್ಕೆ", r"ಬಹು ಆಯ್ಕೆ"],
        "One Word/Sentence": [r"ಒಂದು ವಾಕ್ಯದಲ್ಲಿ", r"ಒಂದೊಂದು ವಾಕ್ಯದಲ್ಲಿ"],
        "Short Answer": [r"ಎರಡು-ಮೂರು", r"ಸಂಕ್ಷಿಪ್ತವಾಗಿ", r"ಮೂರು-ನಾಲ್ಕು"],
        "Long Answer": [r"ಎಂಟು-ಹತ್ತು", r"ಐದು-ಆರು", r"ವಿವರಿಸಿ", r"ಟಿಪ್ಪಣಿ"],
        "Grammar/Vocabulary": [r"ವಿಭಕ್ತಿ", r"ಸಂಧಿ", r"ಸಮಾಸ", r"ಅಲಂಕಾರ", r"ತತ್ಸಮ", r"ತದ್ಭವ", r"ಪದಗಳ ಅರ್ಥ", r"ವ್ಯಾಕರಣ", r"ವಿರುದ್ಧಾರ್ಥಕ", r"ಸ್ವಂತ ವಾಕ್ಯ"],
        "Activity/Project": [r"ಚಟುವಟಿಕೆ", r"ಪೂರಕ ಓದು"]
    }

    current_question = ""
    current_choices = []

    for line in lines:
        line = clean_text(line)
        if not line or len(line) < 2:
            continue

        # --- 1. Detect Question Type Changes (Headers) ---
        found_header = False

        # Check if line contains a keyword indicating a new section
        for section, patterns in header_patterns.items():
            if any(re.search(p, line) for p in patterns):
                # Only switch if the line is short (likely a header) or starts with Roman Numeral
                if len(line) < 100 or re.match(r'^[IVX]+[\.]', line):
                    current_section = section
                    found_header = True
                    break

        if found_header:
            # If we were building a question, save it before switching sections
            if current_question:
                rows.append({
                    "Grade": grade,
                    "Question Type": current_section, # Assigns previous type
                    "Question": current_question,
                    "Choices": " | ".join(current_choices) if current_choices else ""
                })
                current_question = ""
                current_choices = []
            continue

        # --- 2. Detect New Question Start ---
        # Matches: 1. or ೧. or 1)
        if re.match(number_pattern, line):
            # Save previous question
            if current_question:
                rows.append({
                    "Grade": grade,
                    "Question Type": current_section,
                    "Question": current_question,
                    "Choices": " | ".join(current_choices) if current_choices else ""
                })

            # Reset for new question
            current_question = ""
            current_choices = []

            # Remove the numbering (1. / ೧.) to clean the question text
            parts = re.split(r'[\.\)]\s+', line, 1)
            if len(parts) > 1:
                current_question = parts[1].strip()
            else:
                current_question = line

        # --- 3. Detect Options (Specific to MCQ) ---
        # Sometimes options are labeled a), b) or ಅ), ಆ)
        elif current_section == "MCQ" and (re.match(r'^[a-d\u0C85-\u0C88][\.\)]', line) or len(line) < 30):
             # If it's an MCQ section and line looks like an option
             current_choices.append(line)

        # --- 4. Continuation of previous line ---
        else:
            if current_question:
                # Append to existing question text
                current_question += " " + line

    # Save the very last entry
    if current_question:
        rows.append({
            "Grade": grade,
            "Question Type": current_section,
            "Question": current_question,
            "Choices": " | ".join(current_choices) if current_choices else ""
        })

    # Write to CSV
    keys = ["Grade", "Question Type", "Question", "Choices"]
    # 'utf-8-sig' allows Excel to open Kannada files correctly
    with open(output_file, 'w', newline='', encoding='utf-8-sig') as output_csv:
        dict_writer = csv.DictWriter(output_csv, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(rows)

    print(f"Completed! Saved to {output_file}")

# --- UPDATE FILENAME HERE ---
input_filename = "/content/drive/MyDrive/final_kannada_onlyq/8th Kan FL  Part-1 2025-26_EXERCISES.txt"
# Process
if os.path.exists(input_filename):
    process_kannada_file(input_filename, "kannada_dataset.csv")
else:
    print(f"File {input_filename} not found.")

Processing /content/drive/MyDrive/final_kannada_onlyq/8th Kan FL  Part-1 2025-26_EXERCISES.txt for Grade: 8...
Completed! Saved to kannada_dataset.csv


In [ ]:
import re
import csv
import os

def clean_text(text):
    # Remove junk specific to Kannada PDF extractions
    noise = ["PUBLISHED", "NOT TO BE", "KTBS", "PDF", "Government of Karnataka", "Page", "_______", "======="]
    for n in noise:
        text = text.replace(n, "")
    return text.strip()

def process_kannada_file(input_file, output_file):
    rows = []

    # Grade Detection
    grade = "Unknown"
    match = re.search(r'(\d{1,2})', input_file)
    if match: grade = match.group(1)

    # 1. Define Regex Patterns
    # Header: Looks for keywords, but MUST be a short line (to avoid false positives in long text)
    header_patterns = {
        "Match the Following": [r"ಹೊಂದಿಸಿ"],
        "Fill in the Blanks": [r"ಬಿಟ್ಟ ಸ್ಥಳ", r"ಖಾಲಿಬಿಟ್ಟ", r"ಸೂಕ್ತ ಪದ"],
        "MCQ": [r"ಆರಿಸಿ ಬರೆಯಿರಿ", r"ಸರಿಯಾದ ಉತ್ತರ", r"ಬಹು ಆಯ್ಕೆ"],
        "Grammar": [r"ವ್ಯಾಕರಣ", r"ಸಂಧಿ", r"ಸಮಾಸ", r"ತತ್ಸಮ", r"ತದ್ಭವ", r"ವಿಭಕ್ತಿ", r"ಪ್ರತ್ಯಯ", r"ಅಲಂಕಾರ", r"ಛಂದಸ್ಸು"],
        "Short Answer": [r"ಒಂದು ವಾಕ್ಯದಲ್ಲಿ", r"ಎರಡು-ಮೂರು", r"ವಾಕ್ಯಗಳಲ್ಲಿ ಉತ್ತರಿಸಿ"],
        "Long Answer": [r"ಎಂಟು-ಹತ್ತು", r"ವಿವರಿಸಿ", r"ಟಿಪ್ಪಣಿ", r"ಸಂದರ್ಭಸಹಿತ", r"ನಾಲ್ಕು-ಐದು"],
    }

    # Question Start: 1. or 1) or ೧. or ೧)
    # Excludes years like 1947. by checking it's at start of line
    q_start_pattern = re.compile(r'^(\d+|[\u0C66-\u0C6F]+)[\.\)]\s+')

    # Options: a) b) or ಅ) ಆ)
    opt_pattern = re.compile(r'^([a-d]|[\u0C85-\u0C88])[\.\)]\s+')

    current_section = "Textbook Content"
    current_question = ""
    current_choices = []
    current_instruction = "" # Stores instructions like "Split the Sandhi:"

    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        raw_line = line
        line = clean_text(line)

        if not line: continue

        # --- A. Check for Section Headers ---
        # Rule: Must match keyword AND be a short line (e.g., < 80 chars) to be a header
        is_header = False
        if len(line) < 80:
            for section, patterns in header_patterns.items():
                if any(re.search(p, line) for p in patterns):
                    current_section = section
                    # If it's a Grammar header, it might be an instruction like "Split these words"
                    # We save it to append to questions later
                    if section == "Grammar" or section == "Match the Following":
                        current_instruction = line
                    else:
                        current_instruction = ""
                    is_header = True
                    break

        if is_header:
            # Save pending question before switching
            if current_question:
                rows.append([grade, current_section, current_question, " | ".join(current_choices)])
                current_question = ""
                current_choices = []
            continue

        # --- B. Check for New Question Start ---
        if q_start_pattern.match(line):
            # Save previous
            if current_question:
                rows.append([grade, current_section, current_question, " | ".join(current_choices)])

            current_choices = []

            # Remove the number numbering to clean text
            text_without_num = q_start_pattern.sub('', line)

            # If we have a running instruction (e.g. "Split Sandhi"), prepend it for clarity
            if current_instruction and current_section in ["Grammar", "Match the Following"]:
                current_question = f"[{current_instruction}] {text_without_num}"
            else:
                current_question = text_without_num

        # --- C. Check for Options (MCQ) ---
        elif (current_section == "MCQ" or len(current_choices) > 0) and opt_pattern.match(line):
            current_choices.append(line)

        # --- D. Continuation / Formatting Fixes ---
        else:
            if current_question:
                # Formatting Logic:
                # If previous question ended with hyphen, join directly. Else add space.
                if current_question.endswith("-") or current_question.endswith("–"):
                    current_question = current_question[:-1] + line
                else:
                    # Detect if this line is just a broken sentence part
                    current_question += " " + line

    # Save last entry
    if current_question:
        rows.append([grade, current_section, current_question, " | ".join(current_choices)])

    # Output
    with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["Grade", "Question Type", "Question", "Choices"])
        writer.writerows(rows)
    print(f"Kannada processing done: {output_file}")

# Usage
process_kannada_file("/content/drive/MyDrive/kannadatxt/6th Kan FL Part-1 2025-26_EXERCISES-n.txt.txt", "kannada_fixed.csv")

Kannada processing done: kannada_fixed.csv


In [ ]:
import os
import re
import csv
import glob

# ==========================================
# 1. SHARED UTILITIES
# ==========================================

def get_grade_from_filename(filename):
    """Extracts grade number (1-10) from filename."""
    match = re.search(r'(?:Class|Grade|Std|Standard|Part)?[\s_-]*(\d{1,2})', filename, re.IGNORECASE)
    if match:
        return match.group(1)
    return "Unknown"

def clean_text(text):
    """Removes noise common in extracted PDFs."""
    noise = [
        "PUBLISHED", "NOT TO BE", "KTBS", "PDF", "Government of Karnataka",
        "Page", "_______", "=======", "Samacheer", "Tamil Medium", "Textbook",
        "START OF FILE"
    ]
    for n in noise:
        text = text.replace(n, "")
    return text.strip()

# ==========================================
# 2. KANNADA PROCESSING LOGIC
# ==========================================

def process_kannada_folder(input_folder, output_csv_path):
    print(f"--- Starting Kannada Processing in: {input_folder} ---")

    # Check if folder exists
    if not os.path.exists(input_folder):
        print(f"Error: Folder not found: {input_folder}")
        return

    all_rows = []
    files = glob.glob(os.path.join(input_folder, "*.txt"))

    # Regex Patterns
    header_patterns = {
        "Match the Following": [r"ಹೊಂದಿಸಿ"],
        "Fill in the Blanks": [r"ಬಿಟ್ಟ ಸ್ಥಳ", r"ಖಾಲಿಬಿಟ್ಟ", r"ಸೂಕ್ತ ಪದ"],
        "MCQ": [r"ಆರಿಸಿ ಬರೆಯಿರಿ", r"ಸರಿಯಾದ ಉತ್ತರ", r"ಬಹು ಆಯ್ಕೆ"],
        "Grammar": [r"ವ್ಯಾಕರಣ", r"ಸಂಧಿ", r"ಸಮಾಸ", r"ತತ್ಸಮ", r"ತದ್ಭವ", r"ವಿಭಕ್ತಿ", r"ಪ್ರತ್ಯಯ", r"ಅಲಂಕಾರ", r"ಛಂದಸ್ಸು"],
        "Short Answer": [r"ಒಂದು ವಾಕ್ಯದಲ್ಲಿ", r"ಎರಡು-ಮೂರು", r"ವಾಕ್ಯಗಳಲ್ಲಿ ಉತ್ತರಿಸಿ"],
        "Long Answer": [r"ಎಂಟು-ಹತ್ತು", r"ವಿವರಿಸಿ", r"ಟಿಪ್ಪಣಿ", r"ಸಂದರ್ಭಸಹಿತ", r"ನಾಲ್ಕು-ಐದು"],
    }

    q_start = re.compile(r'^(\d+|[\u0C66-\u0C6F]+)[\.\)]\s+')
    opt_start = re.compile(r'^([a-d]|[\u0C85-\u0C88])[\.\)]\s+')

    for file_path in files:
        filename = os.path.basename(file_path)
        grade = get_grade_from_filename(filename)
        print(f"Processing: {filename}")

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        current_section = "Textbook Content"
        current_question = ""
        current_choices = []
        current_instruction = ""

        for line in lines:
            line = clean_text(line)
            if not line: continue

            # A. Header Detection
            is_header = False
            if len(line) < 80:
                for section, patterns in header_patterns.items():
                    if any(re.search(p, line) for p in patterns):
                        current_section = section
                        if section in ["Grammar", "Match the Following"]:
                            current_instruction = line
                        else:
                            current_instruction = ""
                        is_header = True
                        break

            if is_header:
                if current_question:
                    all_rows.append([filename, grade, current_section, current_question, " | ".join(current_choices)])
                    current_question = ""
                    current_choices = []
                continue

            # B. Question Start
            if q_start.match(line):
                if current_question:
                    all_rows.append([filename, grade, current_section, current_question, " | ".join(current_choices)])
                current_choices = []

                q_text = q_start.sub('', line)
                if current_instruction and current_section in ["Grammar", "Match the Following"]:
                    current_question = f"[{current_instruction}] {q_text}"
                else:
                    current_question = q_text

            # C. Options
            elif (current_section == "MCQ" or len(current_choices) > 0) and opt_start.match(line):
                current_choices.append(line)

            # D. Continuation
            else:
                if current_question:
                    if current_question.endswith("-") or current_question.endswith("–"):
                        current_question = current_question[:-1] + line
                    else:
                        current_question += " " + line

        # Save last question of file
        if current_question:
            all_rows.append([filename, grade, current_section, current_question, " | ".join(current_choices)])

    # Write CSV
    headers = ["Source File", "Grade", "Question Type", "Question", "Choices"]
    with open(output_csv_path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(all_rows)
    print(f"Done. Saved to {output_csv_path}\n")


# ==========================================
# 3. TAMIL PROCESSING LOGIC
# ==========================================

def process_tamil_folder(input_folder, output_csv_path):
    print(f"--- Starting Tamil Processing in: {input_folder} ---")

    if not os.path.exists(input_folder):
        print(f"Error: Folder not found: {input_folder}")
        return

    all_rows = []
    files = glob.glob(os.path.join(input_folder, "*.txt"))

    # Regex Patterns
    header_patterns = {
        "MCQ": [r"சரியான சொல்லை", r"தெரிவு செய்து", r"பிரித்து எழுது"],
        "Match": [r"பொருத்துக", r"பொருத்தி"],
        "Fill Blanks": [r"கோடிட்ட", r"நிரப்புக"],
        "Grammar": [r"இலக்கணம்", r"மொழியை ஆள்வோம்", r"எதிர்ச்சொல்", r"புணர்ச்சி", r"அணி"],
        "Short Answer": [r"வினாக்களுக்கு", r"விடையளி", r"குருவினா"],
        "Long Answer": [r"நெடுவினா", r"விரிவான", r"கட்டுரை", r"நயம் பாராட்டுக"],
    }

    q_start = re.compile(r'^(\d+|[IVX]+)[\.\)]\s+')
    opt_start = re.compile(r'^([அஆஇஈஉஊஎஏa-d])[\.\)]\s+')

    for file_path in files:
        filename = os.path.basename(file_path)
        grade = get_grade_from_filename(filename)
        print(f"Processing: {filename}")

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        current_section = "General"
        current_question = ""
        current_choices = []

        for line in lines:
            line = clean_text(line)
            if not line: continue

            # A. Header Detection
            is_header = False
            if len(line) < 60:
                for section, patterns in header_patterns.items():
                    if any(re.search(p, line) for p in patterns):
                        current_section = section
                        is_header = True
                        break

            if is_header:
                if current_question:
                    all_rows.append([filename, grade, current_section, current_question, " | ".join(current_choices)])
                    current_question = ""
                    current_choices = []
                continue

            # B. Question Start
            if q_start.match(line):
                if current_question:
                    all_rows.append([filename, grade, current_section, current_question, " | ".join(current_choices)])
                current_choices = []

                if current_section == "Match":
                    current_question = line
                else:
                    current_question = q_start.sub('', line)

            # C. Options
            elif opt_start.match(line):
                current_choices.append(line)

            # D. Match (Special Case: text pairs)
            elif current_section == "Match" and ("-" in line or "=" in line) and len(line) < 60:
                 if current_question:
                     all_rows.append([filename, grade, current_section, current_question, ""])
                 current_question = line

            # E. Continuation
            else:
                if current_question:
                    if not opt_start.match(line):
                        if current_question.endswith("-"):
                             current_question = current_question[:-1] + line
                        else:
                            current_question += " " + line

        if current_question:
            all_rows.append([filename, grade, current_section, current_question, " | ".join(current_choices)])

    # Write CSV
    headers = ["Source File", "Grade", "Question Type", "Question", "Choices"]
    with open(output_csv_path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(all_rows)
    print(f"Done. Saved to {output_csv_path}\n")


# ==========================================
# 4. CONFIGURATION & EXECUTION
# ==========================================

# ---------------------------------------------------------
# UPDATE THESE PATHS TO MATCH YOUR FOLDERS
# ---------------------------------------------------------

# Example: r"C:\Users\Name\Desktop\Kannada_Txt_Files"
KANNADA_FOLDER = r"/content/drive/MyDrive/kannadatxt"
TAMIL_FOLDER   = r"/content/drive/MyDrive/output_questions_tam"

# Output CSV names
KANNADA_OUTPUT = "Kannada_Questions.csv"
TAMIL_OUTPUT   = "Tamil_Questions.csv"

# Run the functions
if __name__ == "__main__":
    process_kannada_folder(KANNADA_FOLDER, KANNADA_OUTPUT)
    process_tamil_folder(TAMIL_FOLDER, TAMIL_OUTPUT)

--- Starting Kannada Processing in: /content/drive/MyDrive/kannadatxt ---
Processing: 8th Kan FL  Part-1 2025-26_EXERCISES-n.txt.txt
Processing: 7th Kan  FL  Part-1 2025-26_EXERCISES-n.txt.txt
Processing: 6th Kan FL Part-1 2025-26_EXERCISES-n.txt.txt
Processing: 9th  Kan  FL Part- 1 2025-26_EXERCISES-n.txt.txt
Processing: 5th Kan FL Part-1  2025 -26_EXERCISES-n.txt.txt
Processing: 4th Kan FL Part-1  2025-26_EXERCISES-n.txt.txt
Processing: 10th Kan FL  Part-1 2025-26_EXERCISES-n.txt.txt
Processing: Copy of 2nd  Kan FL 2025 Part-1 2025-26_EXERCISES.txt
Processing: Copy of 3rd Kan FL  Part-1 2025 -26_EXERCISES.txt
Processing: Copy of 1std Kan FL Part- 1 2025-26_EXERCISES.txt
Done. Saved to Kannada_Questions.csv

--- Starting Tamil Processing in: /content/drive/MyDrive/output_questions_tam ---
Processing: Class_7_Tamil_Tamil_Medium-Term_1-2024_Edition-www.tntextbooks.in_extracted.txt
Processing: Class_5_Tamil_Tamil_Medium-Term_1-2024_Edition-www.tntextbooks.in_extracted.txt
Processing: Std

In [ ]:
!pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
import os
import re
import csv
import glob

def get_files_from_folder(folder_path):
    """Retrieves all .txt files from the specified folder."""
    if not folder_path or not os.path.isdir(folder_path):
        return []
    return glob.glob(os.path.join(folder_path, "*.txt"))

def clean_text(text):
    """Cleans up extra whitespace and newlines."""
    if not text:
        return ""
    return " ".join(text.split())

def is_noise(line):
    """Checks for headers, footers, page numbers, or separator lines."""
    line = line.strip()
    if not line:
        return True

    # Skip separators
    if set(line).issubset(set("-= *_")):
        return True

    # Skip timestamps/dates (common in PDF extracts)
    if re.search(r'\d{1,2}/\d{1,2}/\d{4}', line):
        return True

    # Skip obvious Page markers or file info
    if re.match(r'^(Page|P\.|பக்கம்)\s*\d+', line, re.IGNORECASE):
        return True
    if "7th Std" in line or "START OF FILE" in line:
        return True

    return False

def is_new_question_start(line):
    """
    Detects if a line starts with a number followed by a dot or bracket.
    Handles Arabic (1.), Kannada (೧.), and Tamil (௧.) numerals.
    """
    # Pattern: Start of line, Optional whitespace, (Digits), (. or ) or -), whitespace
    # Kannada Range: \u0CE6-\u0CEF
    # Tamil Range: \u0BE6-\u0BEF
    pattern = r'^\s*(\d+|[\u0CE6-\u0CEF]+|[\u0BE6-\u0BEF]+)[\.\)\-]\s+'
    return re.match(pattern, line)

def extract_choices_inline(text):
    """
    Attempts to extract MCQ choices if they are embedded in the text
    """
    choices = []

    # Check for Kannada/English style brackets at the end: (A, B, C, D)
    bracket_match = re.search(r'\(([^)]+,[^)]+)\)$', text)
    if bracket_match:
        content = bracket_match.group(1)
        choices = [c.strip() for c in content.split(',')]
        text = text.replace(bracket_match.group(0), "").strip()
        return text, choices

    # Check for Tamil style inline options: அ) ... ஆ) ...
    tamil_option_pattern = r'(?:\s|^)([அஆஇஈ]|\u0B85|\u0B86|\u0B87|\u0B88)[\)]'
    if re.search(tamil_option_pattern, text):
        parts = re.split(tamil_option_pattern, text)
        if len(parts) > 1:
            question_part = parts[0].strip()
            extracted_opts = []
            for i in range(1, len(parts), 2):
                label = parts[i]
                val = parts[i+1] if i+1 < len(parts) else ""
                extracted_opts.append(f"{label}) {val.strip()}")
            return question_part, extracted_opts

    return text, choices

def process_file(filepath):
    """
    Reads a single file and extracts questions/choices.
    """
    extracted_data = []
    filename = os.path.basename(filepath)

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    current_question = ""
    current_choices = []

    for line in lines:
        if is_noise(line):
            continue

        if is_new_question_start(line):
            # Save previous question
            if current_question:
                final_q, inline_choices = extract_choices_inline(current_question)
                if not current_choices and inline_choices:
                    current_choices = inline_choices

                extracted_data.append({
                    "Source_File": filename,
                    "Question": final_q,
                    "Choices": " | ".join(current_choices) if current_choices else ""
                })

            # Start new question
            current_question = clean_text(line)
            current_choices = []

        else:
            cleaned_line = clean_text(line)
            # Check for options on new lines like a), b) or அ), ஆ)
            is_option = re.match(r'^\s*([a-zA-Z]|[அஆஇஈ])[\)\.]\s+', line)

            if is_option and current_question:
                current_choices.append(cleaned_line)
            elif current_question:
                current_question += " " + cleaned_line

    # Save the last item
    if current_question:
        final_q, inline_choices = extract_choices_inline(current_question)
        if not current_choices and inline_choices:
            current_choices = inline_choices

        extracted_data.append({
            "Source_File": filename,
            "Question": final_q,
            "Choices": " | ".join(current_choices) if current_choices else ""
        })

    return extracted_data

def process_folder_to_csv(input_folder, output_csv_path, language_name):
    """
    Helper function to process a whole folder and write to a specific CSV.
    """
    if not input_folder or not os.path.isdir(input_folder):
        print(f"Skipping {language_name}: Folder path not provided or invalid.")
        return

    txt_files = get_files_from_folder(input_folder)
    print(f"\nProcessing {language_name} files from: {input_folder}")
    print(f"Found {len(txt_files)} files.")

    all_questions = []

    for txt_file in txt_files:
        print(f"  - Reading: {os.path.basename(txt_file)}")
        try:
            file_data = process_file(txt_file)
            all_questions.extend(file_data)
        except Exception as e:
            print(f"  ! Error processing {txt_file}: {e}")

    if all_questions:
        keys = ["Source_File", "Question", "Choices"]
        with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(all_questions)
        print(f"Success! Extracted {len(all_questions)} questions to '{output_csv_path}'.")
    else:
        print(f"No questions extracted for {language_name}.")

def main():
    # --- CONFIGURATION ---
    # 1. Input Folders
    kannada_folder = input("Enter the path to the KANNADA text files folder: ").strip()
    tamil_folder = input("Enter the path to the TAMIL text files folder: ").strip()

    # 2. Output Paths
    output_kannada = "/content/drive/MyDrive/extracted_questions_kannada.csv"
    output_tamil = "/content/drive/MyDrive/extracted_questions_tamil.csv"
    # ---------------------

    # Process Kannada
    process_folder_to_csv(kannada_folder, output_kannada, "Kannada")

    # Process Tamil
    process_folder_to_csv(tamil_folder, output_tamil, "Tamil")

if __name__ == "__main__":
    main()

Enter the path to the KANNADA text files folder: /content/drive/MyDrive/kannadatxt
Enter the path to the TAMIL text files folder: /content/drive/MyDrive/output_questions_tam

Processing Kannada files from: /content/drive/MyDrive/kannadatxt
Found 10 files.
  - Reading: 8th Kan FL  Part-1 2025-26_EXERCISES-n.txt.txt
  - Reading: 7th Kan  FL  Part-1 2025-26_EXERCISES-n.txt.txt
  - Reading: 6th Kan FL Part-1 2025-26_EXERCISES-n.txt.txt
  - Reading: 9th  Kan  FL Part- 1 2025-26_EXERCISES-n.txt.txt
  - Reading: 5th Kan FL Part-1  2025 -26_EXERCISES-n.txt.txt
  - Reading: 4th Kan FL Part-1  2025-26_EXERCISES-n.txt.txt
  - Reading: 10th Kan FL  Part-1 2025-26_EXERCISES-n.txt.txt
  - Reading: Copy of 2nd  Kan FL 2025 Part-1 2025-26_EXERCISES.txt
  - Reading: Copy of 3rd Kan FL  Part-1 2025 -26_EXERCISES.txt
  - Reading: Copy of 1std Kan FL Part- 1 2025-26_EXERCISES.txt
Success! Extracted 1568 questions to '/content/drive/MyDrive/extracted_questions_kannada.csv'.

Processing Tamil files from: /c

In [ ]:
import os
import re
import pandas as pd

# List of keywords that trigger a change in the "Question Heading"
# You can add more phrases here if needed.
SECTION_HEADERS = [
    "சரியான விடையைத்",
    "குறுவினா",
    "சிறுவினா",
    "சிந்தனை வினா",
    "கற்பவை கற்றபின்",
    "பொருத்துக",
    "கோடிட்ட இடத்தை",
    "நயம் அறிக",
    "மதிப்பீடு",
    "கலைச்சொல் அறிவோம்"
]

def clean_line(text):
    """
    Removes dashes, dots, and extra whitespace from start/end.
    Example: '--- கற்பவை கற்றபின் .... ---' becomes 'கற்பவை கற்றபின்'
    """
    # Remove leading/trailing dashes, dots, spaces
    text = re.sub(r'^[\s\-\.]+|[\s\-\.]+$', '', text)
    return text

def is_junk_line(line):
    """Checks for page metadata, dates, or separator lines."""
    if "=====" in line: return True
    # Date/Time patterns
    if re.search(r'\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}:\d{2}', line): return True
    # Page footers/headers
    if re.search(r'^\d+th Std -', line) or re.search(r'indd \d+$', line): return True
    if re.match(r'^பக்கம் \(Page\) \d+$', line): return True
    return False

def check_is_heading(line):
    """
    Checks if the line contains any of the predefined section headers.
    Returns the cleaned heading if found, else None.
    """
    cleaned = clean_line(line)
    for header in SECTION_HEADERS:
        if header in cleaned:
            return cleaned # Return the actual text found (e.g., "சிறுவினா")
    return None

def extract_options(line):
    """Extracts Tamil options (அ), ஆ) ...)"""
    option_pattern = r'([அஆஇஈஉ]\))\s*([^அஆஇஈஉ\n]+)'
    matches = re.findall(option_pattern, line)
    formatted_options = []
    if matches:
        for match in matches:
            formatted_options.append(f"{match[0]} {match[1].strip()}")
        return formatted_options
    return []

def process_text_folder_to_csv(input_folder, output_csv):
    data_rows = []

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            filepath = os.path.join(input_folder, filename)

            with open(filepath, 'r', encoding='utf-8') as file:
                lines = file.readlines()

            # State variables
            current_section = ""     # Holds the current heading (e.g., குறுவினா)
            current_question = ""
            current_options = []

            # Helper to save data
            def save_record():
                nonlocal current_question, current_options
                if current_question:
                    data_rows.append({
                        "Source_File": filename,
                        "Question_Heading": current_section, # New Column
                        "Question": current_question.strip(),
                        "Options": " | ".join(current_options) if current_options else ""
                    })
                current_question = ""
                current_options = []

            for line in lines:
                original_line = line.strip()

                # 1. Skip Junk
                if not original_line or is_junk_line(original_line):
                    continue

                # 2. Check if this line is a Section Heading
                # (We do this BEFORE checking for numbers, so headings aren't treated as unnumbered questions)
                found_heading = check_is_heading(original_line)
                if found_heading:
                    # If we find a new heading, save the previous question first
                    save_record()
                    # Update the current section state
                    current_section = found_heading
                    continue

                # 3. Check for Options
                found_opts = extract_options(original_line)
                if found_opts:
                    current_options.extend(found_opts)
                    continue

                # 4. Check for Numbered Questions (Starts with 1., 2., etc.)
                if re.match(r'^\d+\.', original_line):
                    save_record()
                    current_question = original_line

                # 5. Check for Unnumbered Text (Treat as new row)
                else:
                    save_record()
                    # It's a text line, likely a question without a number or a statement
                    data_rows.append({
                        "Source_File": filename,
                        "Question_Heading": current_section,
                        "Question": clean_line(original_line), # Clean dashes if any
                        "Options": ""
                    })

            # Save the very last item in the file
            save_record()

    # Create DataFrame
    if data_rows:
        df = pd.DataFrame(data_rows)
        # Reorder columns to look nice
        df = df[["Source_File", "Question_Heading", "Question", "Options"]]

        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"Success! Processed {len(data_rows)} rows.")
        print(f"File saved at: {output_csv}")
    else:
        print("No data found to extract.")

# --- EXECUTION ---
# Replace with your actual folder path
input_folder_path = r'/content/drive/MyDrive/output_questions_tam'
output_csv_path = '/content/drive/MyDrive/ntamil_questions_extracted.csv'

process_text_folder_to_csv(input_folder_path, output_csv_path)

Success! Processed 4604 rows.
File saved at: /content/drive/MyDrive/ntamil_questions_extracted.csv


In [ ]:
import os
import io
import re
import csv
import glob
import cv2
import numpy as np
from google.cloud import vision
from pdf2image import convert_from_path
from PIL import Image

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "/content/drive/MyDrive/crypto-metric-478912-t1-de1b63214133.json"

# 2. Folder containing your input PDFs
INPUT_FOLDER = "/content/drive/MyDrive/kannada-tb"

# 3. Output filename
OUTPUT_CSV = "/content/drive/MyDrive/newkannada_questions_cleaned.csv"

# ================= LOGIC CONSTANTS =================
START_MARKERS = [r'ಅಭ್ಯಾಸ', r'ಅಭ್ಯಾಸಗಳು']

# Removed 'ವ್ಯಾಕರಣ' so grammar is captured.
END_MARKERS = [
    r'^ಪಾಠ\s+[0-9೦-೯]+',
    r'^ಪದ್ಯ\s+[0-9೦-೯]+',
    r'^ಗದ್ಯ\s+[0-9೦-೯]+',
    r'^ಘಟಕ\s+[0-9೦-೯]+',
    r'^ಪೂರಕ\s+ಓದು',
    r'^ಯೋಜನೆ',
]

SECTION_HEADER_PATTERN = re.compile(r'^\s*([ಅ-ಋa-zA-Z]+)[\.\)]\s+(.*)')
QUESTION_PATTERN = re.compile(r'^\s*([\d0-9೦-೯]+)[\.\)]\s+(.*)')
OPTION_PATTERN = re.compile(r'[\(\[]([ಅ-ಋa-dA-D]+)[\)\]]\s+([^(\[]+)')

# Specific patterns to remove from text if they survive image cleaning
WATERMARK_TEXT_PATTERNS = [
    r'@KTBS',
    r'NOT TO BE REPUBLISHED',
    r'be republished',
    r'Govt of Karnataka',
    r'Government of Karnataka'
]

# ===============================================

def preprocess_image(pil_image):
    """
    Takes a PIL image, converts it to OpenCV format,
    and applies thresholding to remove light watermarks.
    """
    # 1. Convert PIL image to numpy array (OpenCV format)
    img_array = np.array(pil_image)

    # 2. Convert to Grayscale
    # (Handle RGB vs RGBA images)
    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array # Already grayscale

    # 3. Apply Thresholding (Binarization)
    # Any pixel lighter than 180 becomes 255 (White).
    # Any pixel darker than 180 becomes 0 (Black).
    # Since watermarks are usually light grey (around 200-220), they turn White.
    # Text is usually dark (< 50), so it stays Black.
    _, binary = cv2.threshold(gray, 190, 255, cv2.THRESH_BINARY)

    # 4. Convert back to an image format that Google API accepts
    success, encoded_image = cv2.imencode('.jpg', binary)
    return encoded_image.tobytes()

def get_text_from_pdf(pdf_path):
    client = vision.ImageAnnotatorClient()
    text_data = []

    print(f" Converting {os.path.basename(pdf_path)} to images...")
    try:
        # DPI 300 is optimal for OCR
        images = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        print(f"Error converting PDF: {e}")
        return []

    print(f" Pre-processing images and OCR-ing {len(images)} pages...")
    for i, image in enumerate(images):

        # --- NEW: CLEAN IMAGE BEFORE SENDING TO GOOGLE ---
        cleaned_image_bytes = preprocess_image(image)
        # -------------------------------------------------

        image_obj = vision.Image(content=cleaned_image_bytes)

        response = client.document_text_detection(image=image_obj)
        if response.full_text_annotation:
            text_data.append((i + 1, response.full_text_annotation.text))

    return text_data

def clean_ocr_text(text):
    """ Removes OCR artifacts and specific watermark text patterns. """
    lines = text.split('\n')
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Remove empty or very short lines
        if len(line) < 2:
            continue

        # Remove explicit watermark text using Regex
        is_watermark = False
        for pattern in WATERMARK_TEXT_PATTERNS:
            if re.search(pattern, line, re.IGNORECASE):
                is_watermark = True
                break

        if not is_watermark:
            cleaned_lines.append(line)

    return cleaned_lines

def parse_questions(cleaned_lines, filename):
    extracted_data = []

    is_in_exercise = False
    current_context = "General"

    for line in cleaned_lines:
        # 1. Check for Exercise Start
        if any(re.search(m, line) for m in START_MARKERS):
            is_in_exercise = True
            current_context = "General Exercise"
            continue

        # 2. Check for Chapter End
        if any(re.search(m, line) for m in END_MARKERS):
            is_in_exercise = False
            current_context = "General"
            continue

        if not is_in_exercise:
            continue

        # 3. Detect Section Headers
        header_match = SECTION_HEADER_PATTERN.match(line)
        question_match = QUESTION_PATTERN.match(line)

        if header_match and not question_match:
            current_context = header_match.group(2).strip()
            continue

        # 4. Detect Specific Questions
        if question_match:
            q_num = question_match.group(1)
            q_text = question_match.group(2)

            # --- MCQ PARSING ---
            options = []
            opt_matches = OPTION_PATTERN.findall(q_text)

            clean_q_text = q_text
            if opt_matches:
                clean_q_text = re.split(r'[\(\[]([ಅ-ಋa-dA-D]+)[\)\]]', q_text)[0].strip()
                options = [f"({m[0]}) {m[1].strip()}" for m in opt_matches]

            # --- SINGLE WORD FIX ---
            final_question = clean_q_text
            if len(clean_q_text.split()) < 3 and current_context != "General":
                final_question = f"{current_context}: {clean_q_text}"

            row = {
                'Filename': filename,
                'Section_Context': current_context,
                'Q_No': q_num,
                'Question': final_question,
                'Option_1': options[0] if len(options) > 0 else "",
                'Option_2': options[1] if len(options) > 1 else "",
                'Option_3': options[2] if len(options) > 2 else "",
                'Option_4': options[3] if len(options) > 3 else "",
            }
            extracted_data.append(row)

    return extracted_data

def main():
    # Setup CSV file
    csv_headers = ['Filename', 'Section_Context', 'Q_No', 'Question', 'Option_1', 'Option_2', 'Option_3', 'Option_4']

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=csv_headers)
        writer.writeheader()

        pdf_files = glob.glob(os.path.join(INPUT_FOLDER, "*.pdf"))

        if not pdf_files:
            print(f"No PDFs found in {INPUT_FOLDER}")
            return

        for pdf_path in pdf_files:
            filename = os.path.basename(pdf_path)
            print(f"\nProcessing: {filename}")

            # 1. OCR (With Image Cleaning)
            pages_data = get_text_from_pdf(pdf_path)

            full_text_combined = ""
            for _, text in pages_data:
                full_text_combined += text + "\n"

            # 2. Cleanup (Text based)
            cleaned_lines = clean_ocr_text(full_text_combined)

            # 3. Parse
            questions = parse_questions(cleaned_lines, filename)

            # 4. Save to CSV
            if questions:
                writer.writerows(questions)
                print(f" -> Extracted {len(questions)} questions.")
            else:
                print(" -> No questions found.")

    print(f"\nDone! Data saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()